In [1]:
import sys, importlib
sys.path.insert(0, "../src")
from pyspark.sql import functions as F
import main, m1
importlib.reload(m1)

<module 'm1' from 'c:\\Users\\User\\Documents\\GitHub\\projeto_compar_77655_73107\\compar-spark-docker\\notebooks\\../src\\m1.py'>

In [2]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
    .master("local[*]")
    .appName("dev-m1")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate())

c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
trips = main.load_trips(spark, "../databases", [2025], 1.0)
flagged = m1.flag_invalid(trips)
trips_clean = m1.clean(flagged)

In [4]:
spark.conf.set("spark.sql.session.timeZone", "UTC")
import importlib, o1
importlib.reload(o1)

zones = main.load_zones(spark, "../databases/taxi_zone_lookup.csv")
base = o1.addFarePerKm(trips_clean)
stats1 = o1.bucketStats(base)
flagged1 = o1.flagAnomalies(base, stats1)
rates = o1.anomaly_rates(flagged1, zones)
rates.show(50, truncate=False)

+--------------+------------+----------+-------+---------+---------------------+
|pickup_borough|payment_type|is_weekend|trips  |anomalies|anomaly_rate         |
+--------------+------------+----------+-------+---------+---------------------+
|Bronx         |0           |false     |25361  |546      |0.02152911951421474  |
|Bronx         |0           |true      |18794  |185      |0.009843567095881665 |
|Bronx         |1           |false     |13722  |277      |0.020186561725695964 |
|Bronx         |1           |true      |3413   |66       |0.019337825959566363 |
|Bronx         |2           |false     |898    |23       |0.025612472160356347 |
|Bronx         |2           |true      |378    |12       |0.031746031746031744 |
|Bronx         |3           |false     |58     |7        |0.1206896551724138   |
|Bronx         |3           |true      |27     |2        |0.07407407407407407  |
|Bronx         |4           |false     |99     |10       |0.10101010101010101  |
|Bronx         |4           

In [5]:
print(flagged1.count(), trips_clean.count())

flagged1.agg(F.avg(F.col("is_anomaly").cast("int")).alias("rate")).show()

stats1.orderBy(F.desc("std_fpk")).show(5)

rates.filter(F.col("pickup_borough").isNull()).show()

stats = m1.cleaning_stats(flagged)
stats.show()

10247954 10247954
+--------------------+
|                rate|
+--------------------+
|0.004019534045527527|
+--------------------+

+------------+----+------------------+------------------+
|PULocationID|hour|          mean_fpk|           std_fpk|
+------------+----+------------------+------------------+
|           1|   7|3492.2084195997236| 3306.869511126619|
|         138|  12|31.029331341778132|2784.9651838286536|
|           1|   5|2074.1636081309994|2704.5438311711073|
|         175|  21| 590.1178059317838|  2624.20819903999|
|           1|  16|1809.7527253307521| 2454.681067242389|
+------------+----+------------------+------------------+
only showing top 5 rows
+--------------+------------+----------+-----+---------+------------+
|pickup_borough|payment_type|is_weekend|trips|anomalies|anomaly_rate|
+--------------+------------+----------+-----+---------+------------+
+--------------+------------+----------+-----+---------+------------+

+----+--------+------------+-----------